<a href="https://colab.research.google.com/github/almahova/iemocap-proj/blob/data-modelling/iemocap_modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IEMOCAP Emotion Recognition — Modelling Pipeline
## RoBERTa Text Recognition implementation

This notebook implements an **Text-Only Transformer pipeline** for 3-class Speech Emotion Recognition (SER) on the IEMOCAP corpus.

| Component | Choice |
|-----------|--------|
| Model | `RoBERTa` |
| Evaluation | Mix the dataset and divide into 80% train and 20% test. We are looking in this training only on text so we can mix the data |
| Loss | |
| Primary metric | **Macro F1-Score** (robust to the ~55 % Negative class imbalance) |
| Classes | `Negative` (0) · `Positive` (1) · `Neutral` (2) |

## RoBERTa Text Recognition implementation guifline for Claud:

Using an existing (trained) text transformation model RoBERT to classift the data into 3 classes: neutral, positive, negative text.
 You need to use the model and put 10% validation, 10% test, and 80% training or 20% test and 80% train.
 You need to pay attention to the fact that there are more negative records, so the weights should be updated accordingly. You need to put a confusion matrix, accuracy, F1 SCORE, and early stopping mechanisms in case of overfitting.

In [ ]:
import io
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2Processor, Wav2Vec2ForSequenceClassification
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report
import soundfile as sf
import torchaudio.transforms as T
from tqdm import tqdm
from transformers import get_linear_schedule_with_warmup
import matplotlib.pyplot as plt
import itertools
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

In [ ]:
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

In [ ]:
from datasets import load_dataset

# Load IEMOCAP from HuggingFace (mirrors EDA notebook)
ds = load_dataset('AbstractTTS/IEMOCAP')
df = ds['train'].to_pandas()

# Emotion grouping — same mapping as the EDA notebook
emotion_group_map = {
    'happy':      'Positive',
    'excited':    'Positive',
    'angry':      'Negative',
    'frustrated': 'Negative',
    'fear':       'Negative',
    'sad':        'Negative',
    'disgust':    'Negative',
    'other':      'Neutral',
    'neutral':    'Neutral',
    'surprise':   'Neutral',
}
df['emotion_group'] = df['major_emotion'].map(emotion_group_map)

# Implement here